# OpenViBE Raw Signal Viewer

This notebook connects to the `openvibeSignal` LSL stream and displays the **EEG signal** — the same waveform shown in OpenViBE's Signal Display box.

### Run the next OpenVibe environment first: `signal_viewer.xml`

## 1. Imports

In [37]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from pylsl import StreamInlet, resolve_byprop
import time

## 2. Connect to `openvibeSignal`

In [38]:
inlet = None

print('Searching for openvibeSignal stream...')
streams = resolve_byprop('name', 'openvibeSignal', timeout=5.0)

if streams:
    inlet = StreamInlet(streams[0])
    info = inlet.info()
    print(f'\n=== CONNECTED ===')
    print(f'Stream Name: {info.name()}')
    print(f'Type:        {info.type()}')
    print(f'Channels:    {info.channel_count()}')
    print(f'Sample Rate: {info.nominal_srate()} Hz')
else:
    print('\nopenvibeSignal NOT FOUND!')
    print('Make sure your OpenVibe environment is running in Designer.')

Searching for openvibeSignal stream...

=== CONNECTED ===
Stream Name: openvibeSignal
Type:        signal
Channels:    1
Sample Rate: 250.0 Hz


## 3. Live Signal Plot

Continuously redraws the plot with the latest signal data, updating it every 0.3 seconds.

In [41]:
if inlet:
    WINDOW_SECONDS = 10
    SAMPLE_RATE = inlet.info().nominal_srate()
    if SAMPLE_RATE <= 0:
        SAMPLE_RATE = 512
    WINDOW_SIZE = int(WINDOW_SECONDS * SAMPLE_RATE)
    n_channels = inlet.info().channel_count()
    
    signal_buffer = np.zeros((n_channels, WINDOW_SIZE))
    time_axis = np.linspace(-WINDOW_SECONDS, 0, WINDOW_SIZE)
    
    try:
        while True:
            # Accumulate data for 0.3 seconds
            t_acc = time.time()
            while time.time() - t_acc < 0.3:
                chunk, _ = inlet.pull_chunk(timeout=0.0)
                if chunk:
                    data = np.array(chunk).T
                    n = data.shape[1]
                    for ch in range(n_channels):
                        signal_buffer[ch] = np.roll(signal_buffer[ch], -n)
                        signal_buffer[ch, -n:] = data[ch]
                time.sleep(0.01)
            
            # Redraw
            clear_output(wait=True)
            
            fig, axes = plt.subplots(n_channels, 1, figsize=(14, 3 * n_channels), sharex=True)
            if n_channels == 1:
                axes = [axes]
            
            for ch in range(n_channels):
                axes[ch].plot(time_axis, signal_buffer[ch], color='cyan', linewidth=0.6)
                axes[ch].set_facecolor('#0a0a2e')
                axes[ch].set_ylabel(f'Ch {ch+1}', color='white', fontsize=11)
                axes[ch].tick_params(colors='white', labelsize=9)
                ymin = np.min(signal_buffer[ch])
                ymax = np.max(signal_buffer[ch])
                margin = max(0.1 * abs(ymax - ymin), 1e-6)
                axes[ch].set_ylim(ymin - margin, ymax + margin)
                for spine in axes[ch].spines.values():
                    spine.set_color('#333')
            
            axes[-1].set_xlabel('Time (seconds)', color='white')
            axes[0].set_title('OpenVibe Signal', color='white', fontsize=14)
            fig.patch.set_facecolor('#0a0a2e')
            plt.tight_layout()
            plt.show()
            
    except KeyboardInterrupt:
        print('Plot stopped.')
else:
    print('No inlet. Run cell 2 first.')

Plot stopped.
